# 1.3 「猜下一字」該配哪個答案？


材料是「貓看狗。」。我們希望看到「貓」就猜「看」，看到「看」就猜「狗」，看到「狗」就猜句號，而不是看到什麼就照抄什麼。

| 目前提供的字 | 貓 | 看 | 狗 |
| --- | --- | --- | --- |
| 希望猜出的下一字 | 看 | 狗 | 。 |

句號後沒有提供下一字，因此本例不把句號當問題。先只看目前一字，後續再擴大能讀的前文；答案始終是目前位置右邊一格。

把原句拆成清單 `s`，`s[:-1]` 取掉最後一項作問題，`s[1:]` 取掉第一項作答案。切片右端不包含在結果裡；`-1` 表示最後一項。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
s = list("貓看狗。")
x = s[:-1]
y = s[1:]
print(x)
print(y)
for current, answer in zip(x, y, strict=True):
    print("看到", current, "下一字", answer)
assert len(x) == len(y) == 3

前兩行分別印 `['貓','看','狗']` 和 `['看','狗','。']`。`zip` 按相同位置配對，再印出表中的三道題；`strict=True` 要求兩份長度相同。圖裡每條向下箭頭就是一題。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_shift.svg")))

這種錯開一格的對齊常叫 **shift**。它沒有修改原文，而是建立兩份切片。若誤把 `y` 也寫成 `s[:-1]`，題目就變成「貓→貓、看→看、狗→狗」；長度仍正確，學的工作卻變了。

練習先手寫「鳥飛。」的問題與答案，再改程式和長度檢查。應得到「鳥→飛、飛→。」兩題。接著故意令 `y=s[:-1]`，看看為什麼長度檢查無法發現照抄問題。

<details>
<summary>補充與重做</summary>

切片背景見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。shift 決定答案配對；當整段文字同時輸入模型時，還需禁止較早位置讀後面，見 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html)。兩項檢查各有用途，不能互相代替。

</details>
